# VoteMatrix - Usage Examples

This notebook demonstrates the use of the `VoteMatrix` class with various configuration options:
- Simple usage example
- Random vs. predefined party preferences
- Different types of voter turnout
- Reproducible results with seed
- Analysis of parties and vote shares

In [ ]:
from ipres import VoteMatrix, Contestant, contestantsFromParties, Parties, ConstituenciesConfig
parties_config = Parties.from_csv("../../data/examples/de_parties.csv")
constituencies_config = ConstituenciesConfig.from_parquet("../../data/examples/de_bundestag_constituencies.parquet")

contestants : list[Contestant] = contestantsFromParties(parties_config.getPartyNames())
ballot = VoteMatrix.generate(constituencies_config, contestants)
figure = ballot.plot_vote_share_pie("Vote distribution in %")

In [ ]:
from ipres.parties import Parties
from ipres import VoteMatrix, ConstituenciesConfig, Contestant, contestantsFromParties
import numpy as np
from IPython.display import display

# Create a small example configuration
M = 5
Smin, Smax = 10_000, 15_000
N = 4

const_cfg = ConstituenciesConfig.from_random(M=M, Smin=Smin, Smax=Smax, average_turnout_percent=70.0)
parties = Parties.from_random(N=N)
party_names = parties.getPartyNames()

display(const_cfg.getConstituencies().head())
display(parties.getParties())

## 1) Default: random preferences and random voter turnout

- `probabilities=None`: generated internally at random (Dirichlet)
- `turnout=None`: random per constituency (beta distribution with moderate spread)

In [ ]:
vote_matrix_std = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,
    rng=None,
    turnout=None,
)

# Display vote matrix (head)
display(vote_matrix_std.getVotes().head())
# Pie chart of vote shares
fig = vote_matrix_std.plot_vote_share_pie(title="Example 1: Random Preferences & Turnout")
#display(fig)

## 2) Predefined party probabilities (probabilities)

Probabilities are provided as a mapping `party name -> percentage (0–100)`. The sum should equal 100; if not, `VoteMatrix` normalises automatically to 100%.

In [ ]:
party_names = parties.getPartyNames()
probs_map = {name: w for name, w in zip(party_names, [40.0, 30.0, 20.0, 10.0])}

vote_matrix_probs = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=probs_map,
    rng=np.random.default_rng(123),  # optional
    turnout=None,
)

display(vote_matrix_probs.getVotes().head())
fig = vote_matrix_probs.plot_vote_share_pie(title="Example 2: Predefined Probabilities")
display(fig)

## 2b) probabilities as a list/sequence (in percent, order of parties)

Instead of a mapping, a sequence (list/tuple) of percentage values in the order of `participating_parties` can be passed. The length of the list must match the number of parties.

In [ ]:
probs_seq = [45.0, 30.0, 15.0, 10.0]  # in percent, sum = 100
vote_matrix_probs_seq = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=probs_seq,
    rng=np.random.default_rng(321),
    turnout=None,
)

display(vote_matrix_probs_seq.getVotes().head())
fig = vote_matrix_probs_seq.plot_vote_share_pie(title="Example 2b: Probabilities as List (in %)")
#display(fig)

## 3) Reproducible randomness via rng (seed)

A passed `rng` (NumPy Generator) ensures reproducible results.

In [ ]:
rng_seeded = np.random.default_rng(42)
vote_matrix_seeded = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,   # random preferences, but reproducible via rng
    rng=rng_seeded,
    turnout=None,
)

display(vote_matrix_seeded.getVotes().head())
fig = vote_matrix_seeded.plot_vote_share_pie(title="Example 3: Reproducible Randomness (rng=42)")
#display(fig)

## 4) Voter turnout as a single percentage value (float)

`turnout=75.0` means: random values with a mean of 75% (beta distribution) are generated per constituency.

In [ ]:
vote_matrix_turnout_float = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,
    rng=np.random.default_rng(7),
    turnout=75.0,  # percent
)

display(vote_matrix_turnout_float.getVotes().head())
fig = vote_matrix_turnout_float.plot_vote_share_pie(title="Example 4: turnout=75% (as float)")
display(fig)

## 5) Voter turnout per constituency as a mapping

`turnout` can also be provided as a mapping `constituency name -> percent` (0–100).

In [ ]:
const_names = const_cfg.getConstituencyNames()
# Example: alternating 60% and 80%
turnout_map = {name: (60.0 if i % 2 == 0 else 80.0) for i, name in enumerate(const_names)}

vote_matrix_turnout_map = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,
    rng=np.random.default_rng(9),
    turnout=turnout_map,
)

display(vote_matrix_turnout_map.getVotes().head())
fig = vote_matrix_turnout_map.plot_vote_share_pie(title="Example 5: turnout as per-constituency mapping")
display(fig)

## 5b) Voter turnout as a list/sequence (in percent)

Alternatively, `turnout` can be provided as a list/sequence of percentage values for each constituency in the order of `constituencies_config.getConstituencyNames()`. The length of the list must match the number of constituencies.

In [ ]:
turnout_seq = [65.0 if i % 2 == 0 else 75.0 for i in range(len(const_names))]
vote_matrix_turnout_seq = VoteMatrix.generate(
    constituencies_config=const_cfg,
    contestants=contestantsFromParties(party_names),
    probabilities=None,
    rng=np.random.default_rng(11),
    turnout=turnout_seq,
)

display(vote_matrix_turnout_seq.getVotes().head())
fig = vote_matrix_turnout_seq.plot_vote_share_pie(title="Example 5b: turnout as list (in %)")
display(fig)